#### Install dependencies

In [ ]:
!pip install kagglehub pandas scikit-learn Pillow numpy

#### Download dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("youcefattallah97/minerals-identification-classification")
print("Path to dataset files:", path)

#### Display dataset folder structure

In [ ]:
import os

for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:3]:
            print(f"  {indent}{f}")

#### Dataset analysis

In [ ]:
import pandas as pd
from pathlib import Path

base_dir = Path(path) / "Minet 5640 Images"

records = [
    {"filepath": str(img), "label": cls.name}
    for cls in sorted(base_dir.iterdir()) if cls.is_dir()
    for img in cls.glob("*.[jJpP][pPnN][gG]*")
]

df = pd.DataFrame(records)
print(f"Total images: {len(df)}")
print(df['label'].value_counts())

#### (Runs only once) Convert image dataset to structured csv

In [ ]:
from PIL import Image
import numpy as np
import os

if os.path.exists("minerals_structured.csv"):
    print("Already exists, skipping.")
else:
    feature_data = [np.array(Image.open(row["filepath"]).convert("RGB").resize((64, 64))).flatten()
                    for i, row in df.iterrows()]

    feature_df = pd.DataFrame(feature_data, columns=[f"pixel_{i}" for i in range(64 * 64 * 3)])
    feature_df.insert(0, "label", df["label"].values)
    feature_df.to_csv("minerals_structured.csv", index=False)
    print(f"Done! Shape: {feature_df.shape}")

#### Feature and Class counts

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_loaded = pd.read_csv("minerals_structured.csv")

X = df_loaded.drop(columns=["label"]).values
le = LabelEncoder()
y = le.fit_transform(df_loaded["label"].values)

print(f"Features: {X.shape}, Labels: {y.shape}")
print(f"Classes: {le.classes_}")

#### Split Train & Test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

#### Apply SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Before:", np.bincount(y_train))
print("After :", np.bincount(y_train_bal))

#### Train all models

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA

rf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
nb  = GaussianNB()
knn = KNeighborsClassifier(n_neighbors=5)
pca = PCA(n_components=0.95, random_state=42) # added this, should now take 5 minutes

X_train_pca = pca.fit_transform(X_train_bal)
X_test_pca  = pca.transform(X_test)

svm = SVC(kernel='rbf', probability=True, random_state=42)

for name, model, Xtr in [
    ("Random Forest", rf,  X_train_bal),
    ("Naive Bayes",   nb,  X_train_bal),
    ("KNN",           knn, X_train_bal),
    ("SVM",           svm, X_train_pca),
]:
    model.fit(Xtr, y_train_bal)
    print(f"{name} training complete!")

#### Evaluate all models

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

for name, model, Xte in [
    ("Random Forest", rf,  X_test),
    ("Naive Bayes",   nb,  X_test),
    ("KNN",           knn, X_test),
    ("SVM",           svm, X_test_pca),
]:
    y_pred = model.predict(Xte)
    print(f"--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

#### Save all models

In [ ]:
import joblib

joblib.dump(rf,  "mineral_rf_model.pkl")
joblib.dump(nb,  "mineral_nb_model.pkl")
joblib.dump(knn, "mineral_knn_model.pkl")
joblib.dump(svm, "mineral_svm_model.pkl")
joblib.dump(pca, "mineral_pca.pkl") # use this for gradio as well
joblib.dump(le,  "label_encoder.pkl")
print("All models saved!")